# Lezione 13 — Test A/B e disegno degli esperimenti

**Corso:** Statistica Applicata con Python · **Blocco 4:** Modelli ed esperimenti
**Tempo stimato:** 105 minuti · **Capitolo del manuale:** `manual/13_ab_testing_diseno_experimentos_IT.md`

## Cosa vedrai in questa lezione

- Perché la **randomizzazione** permette di parlare di causa (in media) e cosa succede quando le persone si autoselezionano.
- **Disegnare** l'esperimento prima di iniziare: effetto minimo rilevante, α, potenza e **dimensione campionaria**, a mano e con `statsmodels`.
- **Analizzare** un test A/B di conversione: test per due proporzioni, intervallo di confidenza della differenza e *lift*.
- Il ***peeking***: guardare i risultati prima del tempo e fermarsi al primo p < 0,05.
- **Controlli di salute**: SRM (sample ratio mismatch) e test A/A.
- Il **paradosso di Simpson** e i **segmenti e le metriche multiple**.

## Prerequisiti

- Lezione 7 (test per due proporzioni, errori I e II, potenza come idea), lezione 9 (chi-quadro e confronti multipli), lezione 6 (intervalli di confidenza).
- Librerie: `numpy`, `pandas`, `matplotlib`, `scipy`, `statsmodels` (`pip install numpy pandas matplotlib scipy statsmodels`; sono già installate in Google Colab).

> **Come usare questo notebook:** esegui le celle in ordine. Prima di ogni esperimento scrivi cosa ti aspetti che accada. Tutti i dati sono sintetici (*esempio illustrativo*) e conosciamo l'effetto vero, per poter verificare quando il metodo funziona e quando no.

---
## Cella 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_effectsize, confint_proportions_2indep
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings("ignore")

rng = np.random.default_rng(42)          # seme fisso: tutti otteniamo gli stessi risultati
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

def n_por_grupo(p1, p2, alfa=0.05, potencia=0.80):
    """Dimensione campionaria per gruppo per confrontare due proporzioni (formula normale, senza arrotondare)."""
    za, zb = stats.norm.ppf(1 - alfa / 2), stats.norm.ppf(potencia)
    return (za + zb) ** 2 * (p1 * (1 - p1) + p2 * (1 - p2)) / (p2 - p1) ** 2

def z_controles(rng, nsim, controles, n_final=3840, p=0.10):
    """Simula nsim esperimenti A/A (senza effetto): statistica z cumulata a ciascuno dei controlli."""
    por = n_final // controles
    xa = rng.binomial(por, p, (nsim, controles)).cumsum(axis=1)
    xb = rng.binomial(por, p, (nsim, controles)).cumsum(axis=1)
    ns = por * np.arange(1, controles + 1)
    pa, pb, pp = xa / ns, xb / ns, (xa + xb) / (2 * ns)
    return (pb - pa) / np.sqrt(pp * (1 - pp) * 2 / ns)

def falsos_positivos(rng, nsim, controles, umbral=1.96):
    """Percentuale di esperimenti A/A dichiarati 'significativi' in almeno un controllo."""
    sig = np.abs(z_controles(rng, nsim, controles)) > umbral
    return sig.any(axis=1).mean() * 100

def z_dos_proporciones(xa, na, xb, nb):
    """Statistica z per due proporzioni con varianza combinata (vettorizzata)."""
    pa, pb, pp = xa / na, xb / nb, (xa + xb) / (na + nb)
    return (pb - pa) / np.sqrt(pp * (1 - pp) * (1 / na + 1 / nb))

print("Librerie e funzioni caricate correttamente.")

---
## 1. Perché randomizzare

Un negozio online vuole sapere se un **nuovo pulsante di pagamento** (trattamento) fa salire la conversione. Costruiamo una popolazione di 200 000 visitatori in cui conosciamo, per ciascuno, cosa farebbe **con** e **senza** il pulsante (cosa che nella vita reale non vediamo mai). Così l'effetto vero è un numero noto e possiamo giudicare due modi di distribuire il pulsante. Inoltre ogni visitatore ha un **interesse pregresso** all'acquisto, che l'azienda non osserva e che aumenta la probabilità di comprare.

### Esperimento guidato 1: assegnazione casuale contro autoselezione

**Prima di eseguire, prevedi:** se lasciamo che i visitatori più interessati usino il pulsante nuovo (autoselezione), l'effetto stimato sarà maggiore, minore o uguale a quello vero? E se lo assegniamo con una moneta?

In [ ]:
rng_pob = np.random.default_rng(42)
M = 200_000
interes = rng_pob.normal(size=M)                       # interesse pregresso (non osservato)
logit_base = -2.2 + 0.9 * interes
u = rng_pob.random(M)
y_sin = (u < 1 / (1 + np.exp(-logit_base))).astype(int)            # compra SENZA il pulsante nuovo
y_con = (u < 1 / (1 + np.exp(-(logit_base + 0.25)))).astype(int)   # compra CON il pulsante nuovo
efecto_verdadero = (y_con - y_sin).mean() * 100

# A) Assegnazione casuale (moneta)
trat_alea = rng_pob.binomial(1, 0.5, M)
y_alea = np.where(trat_alea == 1, y_con, y_sin)
dif_alea = (y_alea[trat_alea == 1].mean() - y_alea[trat_alea == 0].mean()) * 100

# B) Autoselezione: i più interessati usano di più il pulsante
p_uso = 1 / (1 + np.exp(-1.2 * interes))
trat_auto = rng_pob.binomial(1, p_uso)
y_auto = np.where(trat_auto == 1, y_con, y_sin)
dif_auto = (y_auto[trat_auto == 1].mean() - y_auto[trat_auto == 0].mean()) * 100

print(f"Effetto vero (punti percentuali): {efecto_verdadero:.2f}")
print(f"Stimato con randomizzazione:      {dif_alea:.2f}")
print(f"Stimato con autoselezione:       {dif_auto:.2f}")
print(f"Interesse pregresso medio, autoselezione (trattati / non trattati): {interes[trat_auto == 1].mean():.2f} / {interes[trat_auto == 0].mean():.2f}")
print(f"Interesse pregresso medio, randomizzazione (trattati / non trattati): {interes[trat_alea == 1].mean():.2f} / {interes[trat_alea == 0].mean():.2f}")

fig, ax = plt.subplots()
ax.bar(["Vero", "Casuale", "Autoselezione"], [efecto_verdadero, dif_alea, dif_auto], color=["gray", "tab:green", "tab:red"])
ax.set_ylabel("Effetto stimato (punti percentuali)")
ax.set_title("Stesso pulsante, due modi di distribuirlo")
plt.show()

**Cosa osservi?** Con la moneta il gruppo trattato e quello di controllo hanno lo stesso interesse pregresso (differenza ≈ 0) e la stima cade molto vicino all'effetto vero. Con l'autoselezione i trattati erano già più propensi ad acquistare e l'"effetto" risulta circa quattro volte maggiore: è **correlazione**, non causa (come abbiamo visto nella lezione 10). La randomizzazione equilibra, **in media**, tutto ciò che non misuriamo.

---
## 2. Disegnare prima di iniziare: la dimensione campionaria

Decidiamo **prima** di vedere i dati: metrica primaria (conversione), unità di randomizzazione (visitatore), α = 0,05, potenza 80 % ed **effetto minimo rilevante** (MDE): passare dal 10 % al 12 % (+2 punti). La formula, per due proporzioni e gruppi uguali:

`n = (z(1−α/2) + z(potenza))² · [p₁(1−p₁) + p₂(1−p₂)] / (p₂ − p₁)²`

Prima a mano e poi con `statsmodels` (`proportion_effectsize` fornisce la dimensione dell'effetto *h* di Cohen).

In [ ]:
p_base, p_obj = 0.10, 0.12
za = stats.norm.ppf(1 - 0.05 / 2)
zb = stats.norm.ppf(0.80)
suma_var = p_base * (1 - p_base) + p_obj * (1 - p_obj)
n_mano = (za + zb) ** 2 * suma_var / (p_obj - p_base) ** 2
print(f"z(0,975) = {za:.3f}   z(0,80) = {zb:.3f}   (za+zb)^2 = {(za + zb) ** 2:.3f}   somma delle varianze = {suma_var:.4f}")
print(f"n per gruppo a mano: {n_mano:.1f}")

h = proportion_effectsize(p_obj, p_base)
n_sm = NormalIndPower().solve_power(effect_size=h, alpha=0.05, power=0.80, ratio=1)
print(f"h = {h:.4f}   n per gruppo con statsmodels: {n_sm:.1f}")
pot_2000 = NormalIndPower().solve_power(effect_size=h, nobs1=2000, alpha=0.05, ratio=1)
print(f"Potenza se avessimo solo 2 000 per gruppo: {pot_2000 * 100:.1f} %")

La formula a mano e `statsmodels` differiscono di poche unità (usano approssimazioni leggermente diverse): arrotondiamo per eccesso a **3 840 per gruppo**. Con metà dei visitatori la potenza scende a ~53 %: quasi un testa o croce.

### Come cambia n?

**Prima di eseguire, prevedi:** se l'effetto da rilevare **raddoppia** (da +2 a +4 punti), per quanto si moltiplica n? E se si dimezza?

In [ ]:
filas = []
for base, mde in [(0.10, 0.01), (0.10, 0.02), (0.10, 0.03), (0.10, 0.04)]:
    filas.append({"scenario": f"{base:.0%} → {base + mde:.0%}", "MDE (pp)": mde * 100, "n": round(n_por_grupo(base, base + mde))})
tab_mde = pd.DataFrame(filas)
tab_mde["n relativo a +2 pp"] = (tab_mde["n"] / tab_mde.loc[1, "n"]).round(2)
display(tab_mde)

variantes = pd.DataFrame({
    "caso": ["riferimento (10 % → 12 %, α 5 %, potenza 80 %)", "potenza 90 %", "α = 1 %",
             "tasso base 5 % → 6 % (+20 % relativo)", "tasso base 20 % → 24 % (+20 % relativo)"],
    "n": [round(n_por_grupo(0.10, 0.12)), round(n_por_grupo(0.10, 0.12, potencia=0.90)), round(n_por_grupo(0.10, 0.12, alfa=0.01)),
          round(n_por_grupo(0.05, 0.06)), round(n_por_grupo(0.20, 0.24))]})
display(variantes)

mdes = np.linspace(0.01, 0.06, 50)
fig, ax = plt.subplots()
ax.plot(mdes * 100, [n_por_grupo(0.10, 0.10 + m) for m in mdes])
ax.set_yscale("log")
ax.set_xlabel("Effetto minimo da rilevare (punti percentuali)")
ax.set_ylabel("n per gruppo (scala log)")
ax.set_title("Tasso base 10 %: l'effetto piccolo costa carissimo")
plt.show()

**Cosa osservi?** Raddoppiare l'effetto riduce n a **circa un quarto** (n va con 1/effetto²: il rapporto viene 0,27 e non esattamente 0,25 perché cambiano anche le varianze). Passare da +2 a +1 punto lo quadruplica. Esigere più potenza o meno α costa di più; e a parità di miglioramento *relativo*, un tasso base basso richiede molti più visitatori.

### Esperimento guidato 2: verificare la potenza simulando

Ripetiamo 4 000 volte l'esperimento con effetto **reale** di +2 punti e contiamo quante volte il test risulta significativo con n = 3 840 e con n = 2 000. **Prima di eseguire, prevedi** le percentuali.

In [ ]:
rng_pot = np.random.default_rng(5)
for n in (3840, 2000):
    xa = rng_pot.binomial(n, 0.10, 4000)
    xb = rng_pot.binomial(n, 0.12, 4000)
    z = z_dos_proporciones(xa, n, xb, n)
    print(f"n = {n}:  esperimenti significativi (p < 0,05): {(np.abs(z) > 1.96).mean() * 100:.1f} %")

**Cosa osservi?** Con n = 3 840 il test rileva l'effetto circa l'80 % delle volte (questa è la potenza); con 2 000, circa la metà: nell'altro 50 % esiste un effetto reale e il test dice "non significativo". È l'errore di tipo II della lezione 7, ora con un numero.

### Durata

Con 800 visitatori al giorno in totale (*esempio illustrativo*) servono 2 × 3 840 = 7 680 visitatori: 9,6 giorni. Si arrotonda a **settimane complete** (14 giorni), perché il comportamento cambia tra giorni feriali e fine settimana e non conviene interrompere l'esperimento a metà di un ciclo.

---
## 3. Analizzare un test A/B di conversione

Eseguiamo l'esperimento pianificato: 3 840 visitatori per gruppo, con tassi **veri** 10 % (A, controllo) e 12 % (B). I dati sono sintetici, in formato una riga per visitatore.

In [ ]:
n_grupo = 3840
rng_ab = np.random.default_rng(4)
conv_A = rng_ab.random(n_grupo) < 0.10
conv_B = rng_ab.random(n_grupo) < 0.12
datos_ab = pd.DataFrame({"grupo": ["A"] * n_grupo + ["B"] * n_grupo,
                         "convierte": np.concatenate([conv_A, conv_B]).astype(int)})
resumen = datos_ab.groupby("grupo")["convierte"].agg(["sum", "count", "mean"])
display(resumen)

a, b = int(resumen.loc["A", "sum"]), int(resumen.loc["B", "sum"])
pa, pb = a / n_grupo, b / n_grupo
z, p_valor = proportions_ztest([b, a], [n_grupo, n_grupo])
print(f"Differenza B − A: {(pb - pa) * 100:.2f} pp   z = {z:.3f}   p-value = {p_valor:.4f}")

# IC 95 % della differenza: a mano (normale) e con statsmodels (Newcombe, basato su Wilson)
ee = np.sqrt(pa * (1 - pa) / n_grupo + pb * (1 - pb) / n_grupo)
print(f"IC a mano: [{((pb - pa) - 1.96 * ee) * 100:.2f}, {((pb - pa) + 1.96 * ee) * 100:.2f}] pp")
lo, hi = confint_proportions_2indep(b, n_grupo, a, n_grupo, method="newcomb")
print(f"IC Newcombe: [{lo * 100:.2f}, {hi * 100:.2f}] pp")

# Lift (miglioramento relativo): rapporto pB/pA, con il suo IC
lo_r, hi_r = confint_proportions_2indep(b, n_grupo, a, n_grupo, compare="ratio")
print(f"Lift = {(pb / pa - 1) * 100:.1f} %   IC 95 %: [{(lo_r - 1) * 100:.1f} %, {(hi_r - 1) * 100:.1f} %]")

# Tradotto in termini di business: ogni 100 000 visitatori (esempio illustrativo)
print(f"Conversioni extra ogni 100 000 visitatori: {(pb - pa) * 1e5:.0f}   (IC: {((pb - pa) - 1.96 * ee) * 1e5:.0f} a {((pb - pa) + 1.96 * ee) * 1e5:.0f})")

**Cosa osservi?** Il risultato è significativo (p ≈ 0,02), l'IC della differenza non include lo 0 e il lift è di ~16 %. Ma l'effetto osservato (+1,7 pp) è un po' minore dell'MDE pianificato (+2 pp) e l'IC è ampio: va da +0,3 a +3,1 punti. **Significativo non significa importante:** il valore più basso dell'IC potrebbe non ripagare il costo del cambiamento. E se il test non fosse risultato significativo, ciò **non** avrebbe dimostrato che B è uguale ad A: solo che non abbiamo evidenza sufficiente.

---
## 4. Il peeking: guardare prima del tempo

Tentazione classica: controllare il cruscotto ogni giorno e **fermarsi appena p < 0,05**. Ogni sguardo è una nuova occasione di falso positivo, e le occasioni si accumulano. Lo verifichiamo con esperimenti **A/A**: entrambi i gruppi con lo **stesso** tasso reale (10 %), così ogni "vittoria" è per definizione un falso positivo.

### Esperimento guidato 3: falsi positivi con controlli ripetuti

Simuliamo 4 000 esperimenti A/A da 3 840 visitatori per gruppo. In ciascuno guardiamo 1, 2, 5, 10 o 20 volte (a intervalli regolari) e lo dichiariamo "vincitore" se in **almeno un** controllo |z| > 1,96. **Prima di eseguire, prevedi:** con un solo controllo ci aspettiamo il 5 %; e con 10 o 20?

In [ ]:
rng_peek = np.random.default_rng(7)
controles_lista = [1, 2, 5, 10, 20]
tasas = [falsos_positivos(rng_peek, 4000, L) for L in controles_lista]
tab_peek = pd.DataFrame({"controlli": controles_lista, "% di falsi positivi": np.round(tasas, 1)})
display(tab_peek)

fig, ax = plt.subplots()
ax.plot(controles_lista, tasas, "o-", color="tab:red")
ax.axhline(5, color="gray", linestyle="--", label="livello nominale 5 %")
ax.set_xlabel("Numero di controlli")
ax.set_ylabel("Falsi positivi reali (%)")
ax.set_title("Senza effetto reale, guardare e fermarsi «vince» sempre più spesso")
ax.legend()
plt.show()

**Cosa osservi?** Con un controllo il 5 % nominale è rispettato; con 5 controlli si sfiora il 13 %, con 10 quasi il 19 % e con 20 circa il 25 %: **un esperimento su quattro** senza effetto finisce per "vincere". Vediamo ora come appare una di queste traiettorie, e un rimedio.

In [ ]:
z_todos = z_controles(np.random.default_rng(3), 300, 20)
cruza = (np.abs(z_todos) > 1.96).any(axis=1) & (np.abs(z_todos[:, -1]) < 1.96)
idx = int(np.argmax(cruza))              # primo esperimento che supera la soglia ma termina "non significativo"
fig, ax = plt.subplots()
ax.plot(range(1, 21), z_todos[idx], "o-")
ax.axhline(1.96, color="red", linestyle="--"); ax.axhline(-1.96, color="red", linestyle="--")
ax.set_xlabel("Controllo (ciascuno con più dati)")
ax.set_ylabel("z")
ax.set_title("Un esperimento A/A: supera la soglia e termina senza effetto")
plt.show()
print("Supera la soglia in qualche controllo?", bool(cruza[idx]), "  |  Significativo alla fine?", bool(abs(z_todos[idx, -1]) > 1.96))

# Rimedio concettuale: soglia più esigente a ogni controllo (Pocock, 5 controlli: z ≈ 2,413)
print("5 controlli con |z| > 1,96:", round(falsos_positivos(np.random.default_rng(8), 4000, 5), 1), "%")
print("5 controlli con |z| > 2,413:", round(falsos_positivos(np.random.default_rng(8), 4000, 5, umbral=2.413), 1), "%")

**Cosa osservi?** La traiettoria supera la soglia a un certo punto e, con tutti i dati, rientra: un "vincitore" sarebbe stato un falso positivo. Alzare la soglia a ogni controllo (idea dei **test sequenziali** e dell'*alpha spending*) riporta il tasso a ~5 %, ma è un metodo con regole proprie che va definito **prima** dell'esperimento. La soluzione semplice: fissare n in anticipo e analizzare una sola volta alla fine.

---
## 5. Controlli di salute dell'esperimento

Prima di leggere il risultato bisogna verificare che l'esperimento **sia stato eseguito bene**.

### SRM: sample ratio mismatch

Se abbiamo disegnato una ripartizione 50/50, le dimensioni osservate devono assomigliare a 50/50. Si verifica con un **chi-quadro di bontà di adattamento** (lezione 9). Un p-value molto piccolo (soglia abituale: 0,001) indica un problema di assegnazione o di registrazione: il risultato del test non è affidabile anche se risulta "significativo".

In [ ]:
def control_srm(obs, prop_esperada=None):
    obs = np.array(obs)
    esperado = obs.sum() * (np.array(prop_esperada) if prop_esperada is not None else np.full(len(obs), 1 / len(obs)))
    chi2, p = stats.chisquare(obs, f_exp=esperado)
    return chi2, p

for nombre, obs in [("Esperimento X", [20400, 19600]), ("Esperimento Y", [10050, 9950])]:
    chi2, p = control_srm(obs)
    print(f"{nombre}: {obs}  chi2 = {chi2:.2f}  p-value = {p:.5f}  →  {'ALLARME SRM' if p < 0.001 else 'nessun segnale di SRM'}")
# A mano per X: (400²/20000)·2
print("Verifica a mano di X:", 2 * 400 ** 2 / 20000)

**Cosa osservi?** X ha una differenza di solo il 2 % tra i gruppi, ma con 40 000 visitatori è impossibile per caso (p ≈ 0,00006): qualcosa non va (un *bug* nell'assegnazione, bot, un reindirizzamento lento in B…). Bisogna **risolvere la causa**, non «correggere» statisticamente. Invece Y ha una differenza dell'1 % che il caso spiega bene (p ≈ 0,48).

### Test A/A

Prima di fidarsi della piattaforma di esperimenti si eseguono test **A/A** (due gruppi identici): i p-value devono essere uniformi e circa il 5 % deve risultare sotto 0,05.

In [ ]:
rng_aa = np.random.default_rng(11)
xa = rng_aa.binomial(3840, 0.10, 5000)
xb = rng_aa.binomial(3840, 0.10, 5000)
p_aa = 2 * stats.norm.sf(np.abs(z_dos_proporciones(xa, 3840, xb, 3840)))
print(f"Test A/A con p < 0,05: {(p_aa < 0.05).mean() * 100:.1f} %   (atteso ≈ 5 %)")
fig, ax = plt.subplots()
ax.hist(p_aa, bins=10, range=(0, 1), edgecolor="white")
ax.set_xlabel("p-value"); ax.set_ylabel("Numero di test A/A")
ax.set_title("Senza effetto reale, i p-value sono uniformi")
plt.show()

**Cosa osservi?** I p-value si distribuiscono in modo quasi uniforme e circa il 5 % cade sotto 0,05: la pipeline di analisi è sana. Se vedessi un eccesso di valori piccoli, ci sarebbe un problema (misurazione, unità di analisi…). Un terzo controllo, che citiamo soltanto, è l'**effetto novità**: un cambiamento vistoso può rendere di più all'inizio solo perché è nuovo; guardare se l'effetto si mantiene nelle settimane aiuta a individuarlo.

---
## 6. Il paradosso di Simpson

Una piccola tabella verificabile a mano: due segmenti di dispositivo, con il gruppo B che vince in **ciascun** segmento e perde nel **totale**. (*Esempio illustrativo*, numeri inventati per la dimostrazione.)

In [ ]:
tabla = pd.DataFrame({"segmento": ["desktop", "desktop", "mobile", "mobile"],
                      "grupo": ["A", "B", "A", "B"],
                      "usuarios": [300, 100, 100, 300],
                      "conversiones": [90, 33, 5, 18]})
tabla["tasa %"] = (tabla["conversiones"] / tabla["usuarios"] * 100).round(2)
display(tabla)

total = tabla.groupby("grupo")[["usuarios", "conversiones"]].sum()
total["tasa %"] = (total["conversiones"] / total["usuarios"] * 100).round(2)
display(total)

# Confronto standardizzato: stesso peso di ciascun segmento per A e per B (totale utenti del segmento)
pesos = tabla.groupby("segmento")["usuarios"].sum() / tabla["usuarios"].sum()
tasa = tabla.pivot(index="segmento", columns="grupo", values="conversiones") / tabla.pivot(index="segmento", columns="grupo", values="usuarios")
estand = (tasa.mul(pesos, axis=0)).sum() * 100
print("Tassi standardizzati (%):", estand.round(2).to_dict(), "  differenza B − A:", round(estand["B"] - estand["A"], 2), "pp")

print("Composizione degli utenti per gruppo (% desktop):",
      {g: round(tabla[(tabla.grupo == g) & (tabla.segmento == "desktop")]["usuarios"].iloc[0] / 400 * 100) for g in ("A", "B")})

**Cosa osservi?** B converte di più su desktop (33 % contro 30 %) e su mobile (6 % contro 5 %), ma perde nel totale (12,75 % contro 23,75 %). La causa: il 75 % degli utenti di A è desktop, che converte molto, e solo il 25 % di quelli di B. **I gruppi non sono confrontabili**: in un esperimento ben randomizzato quella differenza di composizione non dovrebbe comparire, e se compare è un segnale di guasto (come un SRM per segmento). Confrontando con lo stesso peso per ogni segmento, B vince di +2 pp. Per evitarlo: randomizzare bene, verificare l'equilibrio per segmento, stratificare e analizzare i segmenti con cautela.

---
## 7. Segmenti e metriche multiple

"Non ha vinto in generale, ma guarda quanto va bene nel segmento 7." Con 10 segmenti (o 10 metriche) e α = 0,05, la probabilità di almeno un falso positivo è 1 − 0,95¹⁰ ≈ 40 %. È il problema dei confronti multipli della lezione 9.

### Esperimento guidato 4: 10 segmenti senza effetto

Simuliamo 2 000 esperimenti A/A con 10 segmenti da 1 000 utenti per gruppo. Contiamo quante volte qualche segmento risulta "significativo" con p non corretti e con la correzione di Holm. **Prima di eseguire, prevedi** entrambe le percentuali.

In [ ]:
rng_seg = np.random.default_rng(13)
n_sim, n_seg, n_u = 2000, 10, 1000
xa = rng_seg.binomial(n_u, 0.10, (n_sim, n_seg))
xb = rng_seg.binomial(n_u, 0.10, (n_sim, n_seg))
p_seg = 2 * stats.norm.sf(np.abs(z_dos_proporciones(xa, n_u, xb, n_u)))
alguno_sin_corregir = (p_seg < 0.05).any(axis=1).mean() * 100
alguno_holm = np.mean([multipletests(fila, alpha=0.05, method="holm")[0].any() for fila in p_seg]) * 100
print(f"Teorico 1 − 0,95^10: {(1 - 0.95 ** 10) * 100:.1f} %")
print(f"Qualche segmento significativo, senza correzione: {alguno_sin_corregir:.1f} %")
print(f"Qualche segmento significativo, con Holm: {alguno_holm:.1f} %")

**Cosa osservi?** Senza correzione, circa 4 esperimenti su 10 **senza alcun effetto** presentano qualche segmento "vincitore"; con Holm, ~5 %. Regola pratica: definisci **una** metrica primaria e un elenco breve di metriche di controllo (*guardrail*) prima di iniziare; tratta le analisi per segmento come esplorative, oppure correggile.

---
## 8. Riepilogo

- **Randomizzare** permette di attribuire la differenza al trattamento, in media, se l'esperimento è eseguito bene.
- **Disegna prima:** metrica primaria, MDE, α, potenza e n (`NormalIndPower`, `proportion_effectsize`). Raddoppiare l'effetto riduce n a ~1/4.
- **Analizza una volta sola**, con test per due proporzioni, IC della differenza e lift; significativo ≠ importante; non significativo ≠ "nessun effetto".
- **Peeking**: con 20 controlli il falso positivo reale è intorno al 25 %.
- **Controlli di salute:** SRM (chi-quadro), A/A, effetto novità.
- **Simpson** e **segmenti/metriche multiple**: confronta gruppi confrontabili e correggi i confronti.

Prosegui con la lezione 14: Progetto integrativo. Ora passa al notebook di pratica.